# transformer-from-scratch â€” end-to-end run on Kaggle

Pretrains a ~17M-parameter Transformer on TinyStories, then adapts it to Shakespeare with
LoRA, and collects every number the README reports.

**Before running â€” in the right-hand sidebar:**

1. **Accelerator â†’ GPU T4 x2** (T4 has fp16 tensor cores; P100 does not, and the run is far slower there).
2. **Internet â†’ On** (needed to fetch the repo and the corpora). Both require a phone-verified Kaggle account.

Expected wall-clock: ~15 min data prep, ~30â€“50 min pretraining, ~10 min LoRA.

## 1. Confirm the GPU

This cell fails deliberately if no accelerator is attached â€” a CPU session would run for
hours and produce nothing useful.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU attached. Set Accelerator -> GPU T4 x2 in the right-hand sidebar, "
        "wait for the session to restart, then run this cell again."
    )

name = torch.cuda.get_device_name(0)
print("GPU:", name)
print("memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
if "T4" not in name:
    print(f"NOTE: expected a T4. {name} works, but fp16 throughput may differ a lot.")

In [ ]:
!nvidia-smi

## 2. Install the project

`--no-deps` is deliberate: Kaggle already ships a CUDA-enabled PyTorch, and letting pip
resolve dependencies here can replace it with a build that cannot see the GPU.

In [ ]:
REPO_URL = "https://github.com/VishnuValmiki3/transformer-from-scratch.git"

# Kaggle can carry /kaggle/working over between versions. Wipe every artifact directory so
# a stale checkpoint from an earlier run cannot survive alongside this run's fresh logs.
!rm -rf /kaggle/working/transformer-from-scratch
!rm -rf /kaggle/working/checkpoints /kaggle/working/data /kaggle/working/artifacts.zip
!git clone -q $REPO_URL /kaggle/working/transformer-from-scratch
%cd /kaggle/working/transformer-from-scratch

!pip install -q -e . --no-deps
!pip install -q regex pyyaml

# Record exactly which commit produced this run's numbers.
print("commit under test:")
!git log --oneline -1

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3. Sanity-check the implementation

The full test suite runs on CPU in well under a minute. If anything here fails, stop â€” the
GPU run will only waste quota.

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -q

## 4. Prepare TinyStories

Two bounded sizes, both deliberate:

- `--bpe-sample-chars 30000000` â€” merge statistics saturate early, so learning them from
  ~30MB is indistinguishable from learning them from the whole corpus, and far faster in Python.
- `--max-train-chars 300000000` â€” 10,000 steps x 32 batch x 256 context consumes ~82M tokens.
  ~300MB of text yields well over that, so encoding the remaining ~1.9GB would buy nothing.

In [ ]:
%%time
!python -m tfs.data.prepare_tinystories \
    --data-dir /kaggle/working/data/tinystories \
    --vocab-size 10000 \
    --bpe-sample-chars 30000000 \
    --max-train-chars 300000000 \
    --workers 4

In [ ]:
import numpy as np
for split in ("train", "valid"):
    tokens = np.memmap(f"/kaggle/working/data/tinystories/{split}.bin", dtype=np.uint16, mode="r")
    print(f"{split}: {len(tokens):,} tokens")

## 5. Smoke run (200 steps)

Cheap insurance. This exercises the CUDA and mixed-precision paths, which the CPU tests
never touch, before committing to the full run.

In [ ]:
!python -m tfs.train --config configs/tinystories_small.yaml \
    --train-tokens /kaggle/working/data/tinystories/train.bin \
    --val-tokens /kaggle/working/data/tinystories/valid.bin \
    --out-dir /kaggle/working/checkpoints/smoke \
    --max-steps 200

## 6. Pretrain

If the smoke run's throughput suggests this will overrun your session, lower `max_steps`
rather than killing it midway â€” checkpoints land every 2,000 steps regardless.

In [ ]:
%%time
!python -m tfs.train --config configs/tinystories_small.yaml \
    --train-tokens /kaggle/working/data/tinystories/train.bin \
    --val-tokens /kaggle/working/data/tinystories/valid.bin \
    --out-dir /kaggle/working/checkpoints/tinystories

## 7. Checkpoint fidelity check

Reload the saved checkpoint and re-score it on the validation set using the same protocol
the training loop used. These must agree â€” if they don't, the file on disk isn't the model
that was trained, and every downstream number would be reported against the wrong weights.

In [ ]:
import json
import math

import numpy as np

from tfs.checkpoint import load_config
from tfs.data.dataset import load_tokens
from tfs.generate import load_model_for_inference
from tfs.train import TrainConfig, estimate_loss

CKPT = "/kaggle/working/checkpoints/tinystories/final.pt"

logged = None
with open("/kaggle/working/checkpoints/tinystories/metrics.jsonl") as f:
    for line in f:
        record = json.loads(line)
        if "val_loss" in record:
            logged = record["val_loss"]

cfg = TrainConfig(**load_config(CKPT))
model = load_model_for_inference(CKPT, "cuda")
val = load_tokens("/kaggle/working/data/tinystories/valid.bin")

# Same seed the training loop uses for evaluation, so the windows are identical.
reloaded = estimate_loss(model, val, cfg, "cuda", cfg.eval_batches, np.random.default_rng(cfg.seed + 1))

print(f"logged at end of training : {logged:.6f}  (ppl {math.exp(logged):.3f})")
print(f"reloaded from checkpoint  : {reloaded:.6f}  (ppl {math.exp(reloaded):.3f})")
if abs(reloaded - logged) > 1e-3:
    raise SystemExit(
        f"CHECKPOINT MISMATCH: saved weights score {reloaded:.4f} but training logged "
        f"{logged:.4f}. The checkpoint on disk is not the trained model."
    )
print("OK - the saved checkpoint is the model that was trained.")

## 8. Loss curve

In [ ]:
import json
import math
import matplotlib.pyplot as plt

def read_metrics(path):
    train, val = [], []
    with open(path) as f:
        for line in f:
            record = json.loads(line)
            (val if "val_loss" in record else train).append(record)
    return train, val

train_log, val_log = read_metrics("/kaggle/working/checkpoints/tinystories/metrics.jsonl")

fig, (ax_loss, ax_speed) = plt.subplots(1, 2, figsize=(13, 4.5))
ax_loss.plot([r["step"] for r in train_log], [r["train_loss"] for r in train_log],
             lw=0.8, alpha=0.6, label="train")
if val_log:
    ax_loss.plot([r["step"] for r in val_log], [r["val_loss"] for r in val_log],
                 lw=2, marker="o", ms=3, label="val")
ax_loss.set_xlabel("step"); ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title("Training curve"); ax_loss.legend(); ax_loss.grid(alpha=0.3)

ax_speed.plot([r["step"] for r in train_log], [r["tokens_per_sec"] for r in train_log],
              lw=0.8, color="tab:green")
ax_speed.set_xlabel("step"); ax_speed.set_ylabel("tokens / sec")
ax_speed.set_title("Throughput"); ax_speed.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/training_curve.png", dpi=140)
plt.show()

if val_log:
    best = min(r["val_loss"] for r in val_log)
    print(f"best val loss {best:.4f} | perplexity {math.exp(best):.2f}")
median_speed = sorted(r["tokens_per_sec"] for r in train_log)[len(train_log) // 2]
print(f"median throughput {median_speed:,.0f} tok/s")

## 9. Sample from the pretrained model

In [ ]:
!python -m tfs.generate \
    --checkpoint /kaggle/working/checkpoints/tinystories/final.pt \
    --vocab /kaggle/working/data/tinystories/vocab.pkl \
    --merges /kaggle/working/data/tinystories/merges.pkl \
    --prompt "Once upon a time" --num-samples 3 --max-new-tokens 150

## 10. LoRA fine-tune onto Shakespeare

The style corpus must be encoded with the **pretraining tokenizer** â€” the model's embedding
table is indexed by those exact ids, so a fresh tokenizer would scramble every input.

In [ ]:
!python -m tfs.data.prepare_style_corpus \
    --data-dir /kaggle/working/data/style \
    --vocab /kaggle/working/data/tinystories/vocab.pkl \
    --merges /kaggle/working/data/tinystories/merges.pkl

In [ ]:
%%time
!python -m tfs.finetune_lora --config configs/lora_style_transfer.yaml \
    --base-checkpoint /kaggle/working/checkpoints/tinystories/final.pt \
    --train-tokens /kaggle/working/data/style/train.bin \
    --val-tokens /kaggle/working/data/style/valid.bin \
    --out-dir /kaggle/working/checkpoints/lora_style

## 11. Before / after

Same prompt, same seed â€” the only difference is the adapter. This is the figure the README
leads with, so keep the output.

In [ ]:
PROMPT = "Once upon a time"

print("=" * 70)
print("BASE MODEL (TinyStories)")
print("=" * 70)
!python -m tfs.generate \
    --checkpoint /kaggle/working/checkpoints/tinystories/final.pt \
    --vocab /kaggle/working/data/tinystories/vocab.pkl \
    --merges /kaggle/working/data/tinystories/merges.pkl \
    --prompt "$PROMPT" --seed 0 --max-new-tokens 150

print()
print("=" * 70)
print("+ LoRA ADAPTER (Shakespeare)")
print("=" * 70)
!python -m tfs.generate \
    --checkpoint /kaggle/working/checkpoints/tinystories/final.pt \
    --adapter /kaggle/working/checkpoints/lora_style/adapter.pt \
    --vocab /kaggle/working/data/tinystories/vocab.pkl \
    --merges /kaggle/working/data/tinystories/merges.pkl \
    --prompt "$PROMPT" --seed 0 --max-new-tokens 150

## 12. What the adapter cost and bought

Perplexity for the same base model with and without the adapter, on both domains. The
second row is the one most write-ups omit: adapting hard to a narrow corpus degrades the
original domain, and that trade-off should be measured rather than assumed away.

In [ ]:
import math

import numpy as np
import torch

from tfs.data.dataset import get_batch, load_tokens
from tfs.generate import load_model_for_inference
from tfs.nn import cross_entropy

BASE = "/kaggle/working/checkpoints/tinystories/final.pt"
ADAPTER = "/kaggle/working/checkpoints/lora_style/adapter.pt"

base_model = load_model_for_inference(BASE, "cuda").eval()
lora_model = load_model_for_inference(BASE, "cuda", adapter=ADAPTER).eval()

def mean_loss(model, data, batches=25):
    total = 0.0
    for i in range(batches):
        x, y = get_batch(data, 16, 256, "cuda", np.random.default_rng(1000 + i))
        with torch.no_grad():
            total += cross_entropy(model(x), y).item()
    return total / batches

rows = []
for label, path in [
    ("Shakespeare (adaptation target)", "/kaggle/working/data/style/valid.bin"),
    ("TinyStories (original domain)", "/kaggle/working/data/tinystories/valid.bin"),
]:
    data = load_tokens(path)
    b, l = mean_loss(base_model, data), mean_loss(lora_model, data)
    rows.append((label, math.exp(b), math.exp(l)))

print("| domain | base ppl | + LoRA ppl | change |")
print("|---|---|---|---|")
for label, pb, pl in rows:
    print(f"| {label} | {pb:,.2f} | {pl:,.2f} | {100 * (pl - pb) / pb:+.1f}% |")

## 13. Benchmark summary

In [ ]:
import json
import math
import os

import torch

def summarize(path):
    train, val = [], []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            (val if "val_loss" in r else train).append(r)
    return train, val

pre_train, pre_val = summarize("/kaggle/working/checkpoints/tinystories/metrics.jsonl")
lora_train, lora_val = summarize("/kaggle/working/checkpoints/lora_style/metrics.jsonl")

base_ckpt = torch.load("/kaggle/working/checkpoints/tinystories/final.pt",
                       map_location="cpu", weights_only=False)
adapter = torch.load("/kaggle/working/checkpoints/lora_style/adapter.pt",
                     map_location="cpu", weights_only=False)

# Rebuild from the saved config rather than summing the state dict: a tied model stores
# token_embedding.weight and lm_head.weight as two keys pointing at one tensor, so summing
# keys double-counts the embedding.
from tfs.train import TrainConfig, build_model

total_params = build_model(TrainConfig(**base_ckpt["config"]), "cpu").num_parameters()
adapter_params = sum(v.numel() for v in adapter["lora"].values())
speeds = sorted(r["tokens_per_sec"] for r in pre_train)

rows = [
    ("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"),
    ("Model parameters", f"{total_params:,}"),
    ("Pretraining steps", f"{pre_train[-1]['step']:,}"),
    ("Median throughput", f"{speeds[len(speeds)//2]:,.0f} tok/s"),
    ("Final train loss", f"{pre_train[-1]['train_loss']:.4f}"),
]
if pre_val:
    best = min(r["val_loss"] for r in pre_val)
    rows += [("Best val loss", f"{best:.4f}"), ("Val perplexity", f"{math.exp(best):.2f}")]
rows += [
    ("LoRA trainable params", f"{adapter_params:,} ({100*adapter_params/total_params:.2f}%)"),
    ("Adapter file size", f"{os.path.getsize('/kaggle/working/checkpoints/lora_style/adapter.pt')/1024:,.1f} KB"),
]
if lora_val:
    best_lora = min(r["val_loss"] for r in lora_val)
    rows += [("LoRA val loss (style)", f"{best_lora:.4f}"),
             ("LoRA val perplexity", f"{math.exp(best_lora):.2f}")]

print("| Metric | Value |")
print("|---|---|")
for name, value in rows:
    print(f"| {name} | {value} |")

## 14. Save the artifacts

Kaggle keeps `/kaggle/working` as the notebook's output. Download this archive and copy
`metrics.jsonl`, `training_curve.png`, and the generated samples into the repo's
`benchmarks/` directory.

In [ ]:
!rm -f /kaggle/working/artifacts.zip
!cd /kaggle/working && zip -qr artifacts.zip \
    checkpoints/tinystories/final.pt \
    checkpoints/tinystories/metrics.jsonl \
    checkpoints/lora_style/adapter.pt \
    checkpoints/lora_style/metrics.jsonl \
    training_curve.png \
    data/tinystories/vocab.pkl \
    data/tinystories/merges.pkl
!ls -lh /kaggle/working/artifacts.zip